## PCM_pix — clean runner notebook

Этот ноутбук — **тонкий сценарий**:
- грузит данные из `data/`
- грузит/обучает суррогаты (legacy или new)
- оценивает качество суррогатов
- запускает оптимизацию (PSO / DE / PSO→DE)
- сохраняет результаты в `outputs/<run_name>/`

Старые большие ноутбуки остаются как архив (`3rd art_...`, `to_server_arch-...`).

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- CONFIG (меняй здесь) ---
CFG = {
    "run_name": "2026_03_12",
    "data_dir": "data",

    # dataset
    "wl": 1.55e-6,
    "mesh_names": [
        "Sb2Se3 - amorphous_mesh_sbse_2025.txt",
        "Sb2Se3 - crystal_mesh12_sbse_2025.txt",
    ],

    # surrogate
    "surrogate_mode": "new",   # "legacy" | "new"
    "device": "cpu",
    "epochs": 10000,
    "lr": 1e-3,

    # optimization settings
    "Nn": 11,
    "b_min_m": 50e-9,

    "opt_mode": "hybrid_pso_de",      # "preset" | "pso" | "pso_until" | "de_full" | "hybrid_pso_de"
    "preset_name": "pos_server_2026_03_04_night",

    # hyperopt modes
    # - legacy ключ (общий), оставлен для совместимости
    # - новые ключи позволяют управлять PSO и DE независимо
    #"hyperopt_mode": "run",        # legacy: "auto" | "use_saved" | "run"
    "pso_hyperopt_mode": "use_saved",    # "auto" | "use_saved" | "run"
    "de_hyperopt_mode": "use_saved",     # "auto" | "use_saved" | "run"

    # PSO
    "pso_threshold": 4,
    "pso_max_restarts": 20,
    "pso_n_particles": 3000,
    "pso_iters": 1000,
    "pso_c1": 0.5,
    "pso_c2": 0.3,
    "pso_w": 0.9,

    # DE (full compatibility)
    "de_init_mode": "init_ar",    # "init_ar" | "x0"
    "de_init_N": 1000,
    "de_mutation": 0.99,
    "de_recombination": 0.1,
    "de_maxiter": 15000,
    "de_popsize": 500,
    "de_tol": 1e-12,
    "de_atol": 1e-12,
    "de_polish": True,
    "de_callback_every": 1000,
    "de_use_linear_constraint": False,


}

# optional: reproducibility seed
SEED = 1
np.random.seed(SEED)

In [ ]:
from pcm_pix.run import start_run, save_json

run = start_run(outputs_dir="outputs", run_name=CFG["run_name"])
logger = run.logger

save_json(run.results / "config.json", CFG)
logger.info("main_clean started")

In [ ]:
# --- Build dataset for surrogates ---
from pcm_pix.features import load_mesh_tables, make_nn_dataset

mesh_tables = load_mesh_tables(CFG, base_dir=CFG["data_dir"])
ds = make_nn_dataset(mesh_tables, wl=CFG["wl"])

logger.info("dataset: df=%s data_0=%s data_1=%s", len(ds.df), len(ds.data_0), len(ds.data_1))
logger.info("X_0=%s y_0=%s | X_1=%s y_1=%s", ds.X_0.shape, ds.y_0.shape, ds.X_1.shape, ds.y_1.shape)

In [ ]:
# --- Load / train surrogates ---
from pcm_pix.nn_surrogate import Net, get_device, load_legacy_surrogates, train_or_load_surrogates

# keep legacy unpickle working (torch.load(model) may reference __main__.Net)
Net

if CFG["device"] == "auto":
    CFG["device"] = get_device()

if CFG["surrogate_mode"] == "legacy":
    CFG["legacy_dir"] = CFG["data_dir"]
    sur0, sur1 = load_legacy_surrogates(CFG, device=CFG["device"])
    logger.info("surrogates loaded: legacy")
else:
    sur0, sur1 = train_or_load_surrogates(ds, run, CFG, force_train=False)
    logger.info("surrogates loaded: new-format")

In [ ]:
# --- Evaluate surrogate quality ---
from pcm_pix.metrics import evaluate_surrogate

qa_am = evaluate_surrogate(ds.data_0, sur0, n=5000, seed=SEED, label="am")
qa_cr = evaluate_surrogate(ds.data_1, sur1, n=5000, seed=SEED, label="cr")

logger.info("QA am: %s", qa_am)
logger.info("QA cr: %s", qa_cr)
qa_am, qa_cr

In [ ]:
# --- Presets / solutions catalog ---
from pcm_pix.optimize import save_solution, load_solution

PRESETS = {
    "pos_2026_03_03_example": np.array([
        999.7153881712717, 989.8568611314538, 846.9306855895525, 812.3405141712682,
        357.50476722603656, 999.0189388651343, 853.5246592828234, 979.0165540829283,
        952.2077494627958, 998.2131906719599, 998.0020446893369, 638.3004244094977,
        645.2087664126163, 681.6157122198844, 702.1334382560799, 284.8875621006104,
        882.7549390819086, 520.3952331766168, 442.78456983718775, 875.627636396043,
        683.0421367034168, 725.5677666087739, 305.7428105275045, 302.3913087111928,
        246.68691437766532, 261.130662543832, 25.83901557205877, 397.619402446743,
        17.140592766392103, 140.12532478580698, 735.0979445623718, 211.69540821165236,
        355.55785007186574, 4.522892213428676, 2.827729567363648,
        0.9073471746931481, 0.8273584342733484
    ]),
    "pos_server_2026_03_04_night":np.array([
        9.99613315e+02, 8.86918846e+02, 6.06035395e+02, 8.54046947e+02,
        5.17645193e+02, 8.79810042e+02, 5.65420119e+02, 9.64383427e+02,
        9.97814618e+02, 9.55635573e+02, 9.95016420e+02, 5.99598526e+02,
        8.29227746e+02, 5.39970803e+02, 8.04025215e+02, 4.17264700e+02,
        6.15239459e+02, 3.62347946e+02, 4.20726459e+02, 6.09650053e+02,
        2.69023666e+02, 4.13247590e+02, 3.76852495e+02, 5.40484452e+02,
        1.99690690e+02, 3.57938384e+02, 4.35299951e+01, 2.57512612e+02,
        2.45768992e+02, 2.64399901e+02, 5.09649663e+02, 1.68530071e+02,
        2.33417540e+01, 4.73217432e+00, 3.35695242e+00, 9.56090488e-01,
        8.91457136e-01
    ]),
    "pos_test_2026_03_12": np.array([9.96841358e+02, 9.37302864e+02, 9.57294114e+02, 5.29040371e+02,
        9.77389395e+02, 8.58376681e+02, 8.42285256e+02, 9.33850847e+02,
        7.63770840e+02, 8.68523525e+02, 9.93136720e+02, 7.13942337e+02,
        1.72741142e+02, 4.06342614e+02, 3.83147986e+02, 6.38101254e+02,
        6.60311825e+02, 5.07564561e+02, 8.34434864e+02, 6.96096300e+02,
        6.80798657e+02, 6.45862244e+02, 2.87477815e+02, 7.04086071e+01,
        4.42449666e+01, 2.50460328e+02, 3.48349163e+02, 2.58011572e+02,
        7.10594600e+01, 3.19886408e+02, 2.33876373e+02, 2.56188765e+02,
        3.06062295e+02, 4.67935757e+00, 6.16194898e+00, 8.00000000e-01,
        8.00000000e-01])
        
}

# save presets once (idempotent)
for name, pos in PRESETS.items():
    save_solution(run, name=name, pos=pos, cost=None, meta={"source": "preset"})

logger.info("presets saved to %s", run.results / "solutions")

In [ ]:
# --- PSO hyperopt: run or use saved ---
# Логика:
# - режим берём из CFG["pso_hyperopt_mode"], а если ключа нет — падаем обратно на legacy CFG["hyperopt_mode"]
# - результаты кэшируются в outputs/<run>/results/pso_*.csv
from pcm_pix.hyperopt import load_best_pso_hyperparams, apply_pso_hyperparams, run_pso_hyperopt

pso_mode = CFG.get("pso_hyperopt_mode", CFG.get("hyperopt_mode", "auto"))

if pso_mode == "run":
    run_pso_hyperopt(sur0, sur1, CFG, run)

hp = load_best_pso_hyperparams(run.results)

if pso_mode == "use_saved" and hp is None:
    raise FileNotFoundError("pso_hyperopt_mode=use_saved but no pso_refine.csv / pso_random_search.csv")

if hp is not None and pso_mode in ("auto", "use_saved", "run"):
    apply_pso_hyperparams(CFG, hp)
    logger.info("PSO hyperparams loaded from %s: c1=%s c2=%s w=%s", hp.source, hp.c1, hp.c2, hp.w)
else:
    logger.info("PSO hyperparams unchanged (hp=%s mode=%s)", hp, pso_mode)

In [ ]:
# --- DE hyperopt: run or use saved ---
# Подбираем гиперпараметры SciPy differential_evolution:
# - de_mutation
# - de_recombination
# - de_popsize
# Результаты кэшируются в outputs/<run>/results/de_*.csv
from pcm_pix.hyperopt import load_best_de_hyperparams, apply_de_hyperparams, run_de_hyperopt

de_mode = CFG.get("de_hyperopt_mode", CFG.get("hyperopt_mode", "auto"))

# Важно: DE-hyperopt делает короткие прогоны DE и должен иметь стартовую точку (pos0)
# Берём её из preset (или можете подставить свой pos вручную).
pos0 = PRESETS[CFG["preset_name"]]

if de_mode == "run":
    run_de_hyperopt(sur0, sur1, CFG, run, pos0=pos0)

hp_de = load_best_de_hyperparams(run.results)

if de_mode == "use_saved" and hp_de is None:
    raise FileNotFoundError("de_hyperopt_mode=use_saved but no de_refine.csv / de_random_search.csv")

if hp_de is not None and de_mode in ("auto", "use_saved", "run"):
    apply_de_hyperparams(CFG, hp_de)
    logger.info(
        "DE hyperparams loaded from %s: mutation=%s recombination=%s popsize=%s",
        hp_de.source, hp_de.mutation, hp_de.recombination, hp_de.popsize,
    )
else:
    logger.info("DE hyperparams unchanged (hp_de=%s mode=%s)", hp_de, de_mode)

In [ ]:
CFG['de_mutation'] = 0.99
CFG['de_recombination'] = 0.1 
CFG['de_popsize'] = 60

In [ ]:
import numpy as np
import torch

def run_one(seed: int):
    # фиксируем сида для всего, что можем
    np.random.seed(seed)
    torch.manual_seed(seed)
    try:
        torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

    logger.info("=== RUN seed=%s ===", seed)

    # тут ровно твой блок оптимизации, только возвращаем (pos, cost)
    if CFG["opt_mode"] == "preset":
        pos = PRESETS[CFG["preset_name"]]
        cost = float(f_vec(pos.reshape(1, -1), sur0, sur1, CFG)[0])
        logger.info("using preset %s cost=%.6f", CFG["preset_name"], cost)

    elif CFG["opt_mode"] == "pso":
        best = run_pso(sur0, sur1, CFG, run)
        pos, cost = best.pos, best.cost

    elif CFG["opt_mode"] == "pso_until":
        best = run_pso_until(sur0, sur1, CFG, run)
        pos, cost = best.pos, best.cost

    elif CFG["opt_mode"] == "de_full":
        pos0 = PRESETS[CFG["preset_name"]]
        best = run_de_full(sur0, sur1, CFG, run, pos=pos0)
        pos, cost = best.pos, best.cost

    else:
        best = run_hybrid_pso_de(sur0, sur1, CFG, run)
        pos, cost = best.pos, best.cost

    logger.info("FINAL cost=%.6f", float(cost))
    logger.info("FINAL pos_len=%s", len(pos))

    return np.array(pos), float(cost)

In [ ]:
K = 50  # сколько прогонов
base_seed = 42  # базовый сид, можно взять из CFG или руками

best_cost = np.inf
best_pos = None
best_seed = None

for k in range(K):
    seed = base_seed + k
    pos_k, cost_k = run_one(seed)

    logger.info("RUN %s/%s seed=%s cost=%.6f", k + 1, K, seed, cost_k)

    if cost_k < best_cost:
        best_cost = cost_k
        best_pos = pos_k
        best_seed = seed

logger.info("=== BEST over %s runs: seed=%s cost=%.6f ===", K, best_seed, best_cost)
best_cost, best_seed

In [19]:
# --- Save final solution to catalog ---
from pcm_pix.optimize import save_solution

name = f"final_{CFG['opt_mode']}"
path = save_solution(run, name=name, pos=best_pos, cost=float(best_cost), meta={"cfg": {"opt_mode": CFG["opt_mode"]}})
logger.info("final solution saved: %s", path.name)
path

2026-03-12 15:13:45,959 | INFO | saved solution final_hybrid_pso_de.json
2026-03-12 15:13:45,962 | INFO | final solution saved: final_hybrid_pso_de.json


PosixPath('outputs/2026_03_12/results/solutions/final_hybrid_pso_de.json')